# MCP Lab 2 — Client + Tool-Calling Agent (Solutions)

In Lab 1 you built an MCP server and poked at it from a few throw-away cells. Now
we build a real **client**: a thin layer that opens a session to the server, lists
its capabilities, and wires the tools into an OpenAI tool-calling **agent loop** so
a user's plain-English question turns into MCP tool calls and back into an answer.

## Learning objectives

By the end of this lab you will be able to:

1. Explain the MCP client/server handshake and lifecycle.
2. Open a `ClientSession`, list capabilities, and call tools from Python.
3. Translate MCP tool descriptors into OpenAI tool-calling format.
4. Implement a robust **tool-calling agent loop** that delegates execution to MCP.
5. Use server-provided **prompts** as user-selectable workflows.

## Prerequisites

- Lab 1 completed; `shop_server.py` lives in this folder and runs.
- `OPENAI_API_KEY` set in your environment or in a `.env` file in this folder.
- `shop.db` at `../talk_to_your_data/shop.db`.

## 1. The protocol from the client's view

From the client's perspective, every interaction with an MCP server is:

1. **Launch** the server (subprocess for stdio, or HTTP request for HTTP transport).
2. **Initialize** — JSON-RPC handshake: exchange protocol version, capabilities.
3. **Discover** — `list_tools`, `list_resources`, `list_prompts`.
4. **Use** — `call_tool(name, args)`, `read_resource(uri)`, `get_prompt(name, args)`.
5. **Tear down** — close the session, server process exits.

The `mcp` Python SDK provides `ClientSession` (the protocol object) and `stdio_client`
(a context manager that handles the subprocess + JSON-RPC plumbing). Together they
make the above five steps two lines of Python.

## 2. Setup

In [ ]:
# %pip install -r requirements.txt

In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path

if sys.platform == "win32":
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    except Exception:
        pass

import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
from openai import OpenAI

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"

SERVER_PARAMS = StdioServerParameters(command=sys.executable, args=["shop_server.py"])
assert Path("shop_server.py").exists(), "Build shop_server.py in Lab 1 first."

## 3. A minimal client

Below: a helper that opens a session, runs a coroutine against it, and closes
cleanly. This is the boilerplate we use for every interaction with the server.

In [ ]:
async def with_session(coro_fn):
    """Open a session to the shop server, run `coro_fn(session)`, return the result."""
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await coro_fn(session)


async def list_all(session):
    tools = await session.list_tools()
    resources = await session.list_resources()
    prompts = await session.list_prompts()
    return tools.tools, resources.resources, prompts.prompts


tools, resources, prompts = await with_session(list_all)
print("tools:", [t.name for t in tools])
print("resources:", [str(r.uri) for r in resources])
print("prompts:", [p.name for p in prompts])

## 4. Calling tools directly

Before plugging the server into an agent loop, make sure you can call tools by hand —
agent debugging is *much* easier when you know the underlying call works in isolation.

In [ ]:
async def call_top(session):
    return await session.call_tool("top_products_by_rating", {"limit": 3})


result = await with_session(call_top)
for c in result.content:
    if hasattr(c, "text"):
        print(c.text)

## 5. Bridging MCP tools to OpenAI tool-calling

The OpenAI Chat Completions API accepts a `tools` argument of the form

```json
[{
    "type": "function",
    "function": {
        "name": "...",
        "description": "...",
        "parameters": {<JSON schema>}
    }
}]
```

MCP's `list_tools()` response gives us tools with `.name`, `.description`, and
`.inputSchema` (already a JSON Schema dict). Translation is a one-line shape change.

### TODO 5.1 — `mcp_tool_to_openai`

In [ ]:
def mcp_tool_to_openai(tool) -> dict:
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.inputSchema,
        },
    }


openai_tools = [mcp_tool_to_openai(t) for t in tools]
print(json.dumps(openai_tools[0], indent=2))

## 6. The agent loop

The standard tool-calling pattern, regardless of which provider you use:

1. Send the conversation so far + the tool catalogue to the model.
2. The model either returns a final answer (done) **or** one-or-more tool calls.
3. Execute each tool call. For us "execute" means `session.call_tool(name, args)`.
4. Append the tool results to the conversation as `role: tool` messages.
5. Loop back to (1).

It's a few lines of code, but two details often go wrong:

- **Append the assistant message including its tool_calls field** before the tool
  results. The API uses the `tool_call_id` to match each tool result to the call it
  answered.
- **Stop on something.** Cap iterations so a buggy server-side loop can't run forever.

### TODO 6.1 — `run_agent`

Implement an agent loop with the contract:

```
async def run_agent(session, user_message, history, openai_tools, max_iters=8) -> str
```

- Append the user message to `history`.
- Loop up to `max_iters` times:
  - Call `chat.completions.create(model=MODEL, messages=history, tools=openai_tools)`.
  - Append the assistant message (with `tool_calls` if present) to `history`.
  - If there are no `tool_calls`, return the assistant content.
  - Otherwise, for each tool call: `session.call_tool(...)`, append the result as a
    `role: tool` message.
- If you hit `max_iters`, return a sentinel string explaining you stopped.

In [ ]:
async def run_agent(session, user_message: str, history: list, openai_tools: list, max_iters: int = 8) -> str:
    history.append({"role": "user", "content": user_message})

    for _ in range(max_iters):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=history,
            tools=openai_tools,
            temperature=0,
        )
        msg = resp.choices[0].message

        # Append assistant message (including tool_calls field).
        assistant_entry = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            assistant_entry["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in msg.tool_calls
            ]
        history.append(assistant_entry)

        if not msg.tool_calls:
            return msg.content or ""

        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            tool_result = await session.call_tool(tc.function.name, args)
            text = "".join(getattr(c, "text", "") for c in tool_result.content)
            if tool_result.isError:
                text = f"[tool error] {text}"
            history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": text,
            })

    return "[agent stopped: max iterations reached]"

### Driving the agent

Now we put it all together: one session, one conversation history, several user
turns. Watch how the model picks tools and chains calls.

In [ ]:
SYSTEM_PROMPT = (
    "You are a shop assistant. You answer questions about an e-commerce database by "
    "calling the available tools. Always call a tool when the user asks about products, "
    "orders, or reviews — never guess. Keep replies concise."
)

USER_TURNS = [
    "Which products have the best reviews?",
    "What categories of products do we carry?",
    "Show me the orders for customer 1 and tell me what their favourite category is.",
]


async def chat_session():
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            listing = await session.list_tools()
            openai_tools = [mcp_tool_to_openai(t) for t in listing.tools]

            history = [{"role": "system", "content": SYSTEM_PROMPT}]
            for turn in USER_TURNS:
                print(f"\n>>> user: {turn}")
                reply = await run_agent(session, turn, history, openai_tools)
                print(f"<<< assistant: {reply}")


await chat_session()

## 7. Using server prompts as workflows

Server **prompts** are the part of MCP many tutorials skip, but they're powerful.
Think of them as **user-selectable workflows** the server defines: parameterized
message templates that the user can invoke ("run the customer-summary workflow on
customer 7"), with the agent loop executing whatever tool calls the prompt implies.

Below we fetch the `summarize_orders` prompt from the server and feed it through the
same `run_agent` loop. The server is doing dual duty: providing the *task statement*
(via the prompt) **and** the *capability to do it* (via the tools).

In [ ]:
async def run_summary_for(customer_id: int):
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            listing = await session.list_tools()
            openai_tools = [mcp_tool_to_openai(t) for t in listing.tools]

            prm = await session.get_prompt("summarize_orders", {"customer_id": str(customer_id)})
            # The prompt comes back as a list of messages. We unwrap the user text.
            user_text = "\n\n".join(
                m.content.text for m in prm.messages if hasattr(m.content, "text")
            )

            history = [{"role": "system", "content": "Use the available tools to answer."}]
            return await run_agent(session, user_text, history, openai_tools)


print(await run_summary_for(1))

## 8. Composing multiple servers (briefly)

The same client code talks to *any* MCP server. In real systems you'd run several:
one for your database, one for the filesystem, one for an internal API. The agent
merges their tool catalogues into a single `openai_tools` list and the model picks
across all of them transparently.

The reference filesystem server is published as an npm package
(`@modelcontextprotocol/server-filesystem`) — `StdioServerParameters(command="npx",
args=["@modelcontextprotocol/server-filesystem", "/some/path"])`. Combining it with
your shop server is one extra session and one bigger `openai_tools` list. This is the
composability MCP is built for.

## Summary

You now have a complete MCP-driven agent:

- A real client (`with_session`, `ClientSession`).
- A schema bridge (`mcp_tool_to_openai`).
- A robust agent loop (`run_agent`) that delegates tool execution to MCP.
- A way to invoke server-provided prompt workflows through the same loop.

Across the two labs you saw both ends of the MCP wire — author the server's
contract once, then any client can consume it without extra plumbing.

## Exercises

1. **Tool error recovery.** Modify `run_agent` so when a tool returns `isError`, the
   next iteration includes a system reminder telling the model to recover or ask the
   user. Test on `get_customer_orders(customer_id=999)` (which doesn't exist).
2. **Streaming output.** Replace `chat.completions.create` with the streaming variant
   and print partial tokens as they arrive. Keep tool execution synchronous.
3. **Two servers.** Run your shop server *and* the filesystem MCP server side by
   side. Merge their tools. Ask a question that needs both (e.g. "list product
   categories, then write them to ./categories.txt"). Note how the agent picks
   cross-server.
4. **Add a resource to the conversation.** When the user mentions "the schema",
   fetch `schema://shop` and inject it as a system message before the next agent
   turn. (This is how IDE-style clients surface server resources.)
5. **Cost guardrail.** Sum `response.usage.total_tokens` across the agent loop. If a
   single turn exceeds a budget, abort and return a polite message.